<a href='https://colab.research.google.com/github/Emelecto/QuantLab/blob/main/web/content/cursos/ml/notebooks/c5_l4.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg'/></a>

# C5-L4 · Ridge vs LightGBM
Lineal con freno contra boosting: decide con Sharpe neto (con costos), no con R².

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path
URL = 'https://raw.githubusercontent.com/Emelecto/QuantLab/main/web/content/cursos/ml/data/c5_l4.csv'
try:
    df = pd.read_csv(URL)
    print('Fuente: URL (Colab)')
except Exception as e:
    print('Sin red, uso fallback local:', e)
    for cand in [Path('../data/c5_l4.csv'), Path('data/c5_l4.csv'), Path('c5_l4.csv')]:
        if cand.exists():
            df = pd.read_csv(cand); break
    print('Fuente: local')
print(df.shape)
print(df.head(5).to_string(index=False))

In [ ]:
# Features de mercado + etiqueta direccional con costos
df['ret'] = df['close'].pct_change()
df['rango'] = (df['high']-df['low'])/df['close']
df['vol_z'] = (df['volumen']-df['volumen'].rolling(10).mean())/df['volumen'].rolling(10).std()
for k in (1, 2, 3, 5, 8):
    df[f'lag_{k}'] = df['ret'].shift(k)
df['mom5'] = df['close']/df['close'].shift(5) - 1
data = df.dropna().reset_index(drop=True)
COSTO = 0.001
data['y'] = ((data['ret'].shift(-1) - COSTO) > 0).astype(int)
data = data.iloc[:-1].reset_index(drop=True)
feat = ['rango','vol_z','lag_1','lag_2','lag_3','lag_5','lag_8','mom5']
print('filas:', len(data), 'features:', feat)
assert data[feat].isna().sum().sum() == 0 and len(data) > 100

In [ ]:
# Ridge vs Boosting (LightGBM si esta, si no HistGradientBoosting de sklearn)
from sklearn.linear_model import RidgeClassifier
try:
    import lightgbm as lgb
    def make_gbm():
        return lgb.LGBMClassifier(n_estimators=100, verbose=-1)
    print('Motor GBM: LightGBM')
except Exception as e:
    from sklearn.ensemble import HistGradientBoostingClassifier
    def make_gbm():
        return HistGradientBoostingClassifier(max_iter=100)
    print('Motor GBM: HistGradientBoosting (fallback sklearn).', type(e).__name__)
from sklearn.metrics import accuracy_score
split = int(len(data)*0.7); EMB = 5
X = data[feat].values; y = data['y'].values
m_ridge = RidgeClassifier().fit(X[:split-EMB], y[:split-EMB])
m_gbm = make_gbm().fit(X[:split-EMB], y[:split-EMB])
pr, pg = m_ridge.predict(X[split:]), m_gbm.predict(X[split:])
print(f'acc Ridge={accuracy_score(y[split:], pr):.3f}  GBM={accuracy_score(y[split:], pg):.3f}')

In [ ]:
# Sharpe neto: estrategia = signo predicho x retorno siguiente - costos por cambio de posicion
rets = data['ret'].values[split:]
def sharpe_neto(pred):
    pos = np.where(pred == 1, 1.0, -1.0)
    turnover = np.abs(np.diff(np.r_[0, pos])) / 2  # 1 cuando cambia de lado
    pnl = pos * rets - turnover * COSTO
    return float(pnl.mean()/pnl.std()*np.sqrt(252)) if pnl.std() > 0 else 0.0
sh_r, sh_g = sharpe_neto(pr), sharpe_neto(pg)
print(f'Sharpe neto Ridge={sh_r:.3f}  GBM={sh_g:.3f}')
ganador = 'Ridge' if sh_r >= sh_g else 'GBM'
print('Gana (Sharpe neto):', ganador)

In [ ]:
# Chequeo automatico L4
assert np.isfinite(sh_r) and np.isfinite(sh_g)
assert -5 < sh_r < 5 and -5 < sh_g < 5
assert len(pr) == len(pg) == len(data) - split
print(f'OK L4: Ridge vs GBM comparado, Sharpe neto manda ({ganador})')